# TrafficLens · Phase 1: data preparation (Kaggle)

This notebook remaps the IDD-Detection dataset to the 7 TrafficLens classes, keeps a training subset, and draws the class-distribution charts. It **does not need a GPU** and takes roughly 5–15 minutes.

## Before you run it: one-time setup

1. On kaggle.com, click **Create → New Notebook**. Then **File → Import Notebook** and upload this `.ipynb` file.
2. In the right-hand panel, click **Add Input**, search for **`indian-driving-dataset-detections-yolov11`** (by *redzapdos123*) and click **+** to attach it. This adds nothing to your 20 GB working space.
3. Still in the right-hand panel, under **Session options**:
   - **Accelerator:** `None` (a GPU isn't needed for this step).
   - **Internet:** `On`. The notebook downloads the TrafficLens code from GitHub. If the switch is greyed out, first verify your phone number at kaggle.com → your profile picture → **Settings** → **Phone verification**, then reload this notebook.
## Run it

Click **Run All**. When it finishes:

- Look at the chart and tables at the bottom.
- Download **`phase1_reports.zip`** from the **Output** panel (right side, under `/kaggle/working`) and send me its contents, or just a screenshot of the last three cells.

In [ ]:
# 1. Check that the dataset is attached (prints where its data.yaml is)
import glob
hits = glob.glob("/kaggle/input/**/data.yaml", recursive=False) + glob.glob("/kaggle/input/*/*/*/*/data.yaml") + glob.glob("/kaggle/input/*/*/*/*/*/data.yaml")
print(hits or "No data.yaml found: attach the dataset with 'Add Input' (see the instructions above)")


In [ ]:
# 2. Get the TrafficLens code (change BRANCH only if I ask you to)
import subprocess, sys
BRANCH = "claude/loving-edison-eq4h8f"
REPO_DIR = "/kaggle/working/trafficlens"
subprocess.run(["rm", "-rf", REPO_DIR], check=True)
clone = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                        "https://github.com/NiKeT972/trafficlens.git", REPO_DIR],
                       capture_output=True, text=True)
if clone.returncode != 0:
    raise RuntimeError("Could not download the code from GitHub.\n"
                     "Most likely INTERNET IS OFF: right-hand panel -> Session options -> Internet -> On,\n"
                     "then Run All again.\n\nDetails: " + clone.stderr)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
sys.path.insert(0, f"{REPO_DIR}/src")
print("TrafficLens code ready.")


In [ ]:
# 3. Prepare the dataset: remap classes, subsample train, symlink images, write data.yaml
%cd /kaggle/working/trafficlens
!python -m trafficlens.data.prepare \
    --data-config configs/data.yaml \
    --classes-config configs/classes.yaml \
    --dataset-dir /kaggle/working/datasets/trafficlens_idd \
    --reports-dir /kaggle/working/reports/data

In [ ]:
# 4. Class distribution (prepared dataset)
import json, pandas as pd
from IPython.display import Image, display
from trafficlens.data.stats import imbalance_ratio

R = "/kaggle/working/reports/data"
display(Image(f"{R}/class_distribution.png"))

prepared = pd.read_csv(f"{R}/class_stats_prepared.csv")
source = pd.read_csv(f"{R}/class_stats_source.csv")
print("Instances per class (prepared):")
display(prepared.pivot(index="class", columns="split", values="instances").loc[prepared["class"].unique()])
print("Train before vs after subsampling (instances):")
display(pd.DataFrame({"source_train": source[source.split=="train"].set_index("class")["instances"],
                      "prepared_train": prepared[prepared.split=="train"].set_index("class")["instances"]}))
for s in prepared["split"].unique():
    print(f"imbalance ratio (largest/smallest class), {s}: {imbalance_ratio(prepared, s):.1f}x")

In [ ]:
# 5. Summary + sample file names (I use these to check for near-duplicate frames across splits)
summary = json.load(open(f"{R}/summary.json"))
print(json.dumps(summary, indent=2))

In [ ]:
# 6. Bundle the small report files for download (the dataset itself stays on Kaggle)
!cd /kaggle/working && zip -qr phase1_reports.zip reports/data && ls -lh phase1_reports.zip